# Навчання моделі YOLOv8l для розпізнавання товарів

У цьому ноутбуці виконано підготовку набору даних RPC Dataset та навчання моделі YOLOv8l для задачі автоматичного розпізнавання товарів.  
Початкові анотації датасету подані у форматі COCO JSON, тому перед навчанням вони конвертуються у формат YOLO.  
Після підготовки даних запускається навчання моделі YOLOv8l із використанням попередньо навчених ваг `yolov8l.pt`.

In [ ]:
# Встановлення бібліотеки Ultralytics, яка містить реалізацію моделей YOLOv8
!pip install -U ultralytics

In [ ]:
# Імпорт класу YOLO для завантаження та навчання моделі
from ultralytics import YOLO

In [ ]:
# Встановлення бібліотеки pycocotools для роботи з анотаціями у форматі COCO
!pip install pycocotools

In [ ]:
# Встановлення tqdm для відображення прогресу виконання циклів
!pip install tqdm

In [ ]:
def tqdm(x):
    return x

In [ ]:
# Імпорт бібліотек для роботи з JSON-файлами та файловою системою
import json
import os

## Конвертація анотацій із формату COCO у формат YOLO

RPC Dataset містить анотації у форматі COCO JSON.  
Для навчання YOLOv8 потрібно перетворити їх у формат YOLO, де кожному зображенню відповідає окремий `.txt` файл.

Кожен рядок у файлі розмітки YOLO має такий вигляд:

`class_id x_center y_center width height`

Усі координати нормалізуються відносно ширини та висоти зображення.

In [ ]:
# Функція для конвертації анотацій COCO JSON у формат YOLO
def coco_to_yolo(coco_json, images_dir, labels_dir):
    # Створення папки для збереження YOLO-розмітки
    os.makedirs(labels_dir, exist_ok=True)

    # Зчитування COCO JSON-файлу
    with open(coco_json) as f:
        coco = json.load(f)

    # Формування словника з інформацією про зображення
    images = {img["id"]: img for img in coco["images"]}

    # Перетворення id категорій COCO у послідовні id класів YOLO
    categories = {cat["id"]: i for i, cat in enumerate(coco["categories"])}

    # Групування анотацій за id зображення
    anns_by_image = {}
    for ann in coco["annotations"]:
        anns_by_image.setdefault(ann["image_id"], []).append(ann)

    # Обробка кожного зображення та створення відповідного txt-файлу
    for img_id, img in tqdm(images.items()):
        w, h = img["width"], img["height"]

        label_path = os.path.join(
            labels_dir, img["file_name"].replace(".jpg", ".txt")
        )

        with open(label_path, "w") as f:
            for ann in anns_by_image.get(img_id, []):
                # COCO bbox: x_min, y_min, width, height
                x, y, bw, bh = ann["bbox"]

                # Перетворення координат у формат YOLO
                x_center = (x + bw / 2) / w
                y_center = (y + bh / 2) / h
                bw /= w
                bh /= h

                cls = categories[ann["category_id"]]

                # Запис рядка у форматі YOLO
                f.write(f"{cls} {x_center} {y_center} {bw} {bh}\n")

## Конвертація тренувальної та валідаційної вибірок

На цьому етапі окремо конвертуються анотації для train та validation вибірок.

In [ ]:
# Конвертація анотацій тренувальної вибірки
coco_to_yolo(
    coco_json=r"C:\Product recognition\Dataset\instances_train2019.json",
    images_dir=r"C:\Product recognition\Dataset\train2019",
    labels_dir=r"C:\Product recognition\Dataset_yolo\labels\train"
)

In [ ]:
# Конвертація анотацій валідаційної вибірки
coco_to_yolo(
    coco_json=r"C:\Product recognition\Dataset\instances_val2019.json",
    images_dir=r"C:\Product recognition\Dataset\val2019",
    labels_dir=r"C:\Product recognition\Dataset_yolo\labels\val"
)

## Завантаження моделі YOLOv8l

Для навчання використовується модель `YOLOv8l`.  
Було застосовано підхід transfer learning, тобто навчання починається з попередньо навчених ваг `yolov8l.pt`. Це дозволяє скоротити час навчання та покращити якість детекції.

In [ ]:
# Завантаження попередньо навченої моделі YOLOv8l
model = YOLO("yolov8l.pt")

## Навчання моделі

Навчання виконується на підготовленому наборі даних у форматі YOLO.  
Для покращення розпізнавання дрібних товарів використано збільшений розмір вхідного зображення `960×960`.  
Також застосовано аугментації, ранню зупинку та mixed precision training.

In [ ]:
# Запуск навчання моделі YOLOv8l
model.train(
    data="Dataset_yolo/data.yaml",       # YAML з шляхами до train/val і класами
    epochs=120,                          # кількість епох
    imgsz=960,                           # розмір для дрібних товарів
    batch=8,                             # кількість зображень в одному батчі
    patience=30,                         # рання зупинка
    save_period=10,                      # збереження чекпоінтів
    device=0,                            # GPU
    amp=True,                            # змішане навчання 
    mosaic=True,                         # комбінування зображень
    mixup=True,                          # поєднання зображень
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,   # колірні аугментації 
    degrees=10,                          # поворот
    translate=0.1,                       # зміщення
    scale=0.5,                           # масштабування
    shear=2,                             # скошування
    lr0=0.003                            # початкове значення швидкості навчання
)